[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/notebooks/EN/chapter4_seminar_notebook.ipynb)

# Modelling Financial Markets — Seminar 4: Portfolio Optimization

*Seminar notebook. Daniel Traian Pele, Antoaneta Amza — Bucharest University of Economic Studies.*

- **Part A**: numerical check of the nine derivations.
- **Part B**: backtests and inference on US data and Bitcoin.
- **Part C**: an optimal portfolio of BVB blue chips in RON: does diversification beyond BET pay?
- **[Solved]** problems are fully worked out (code and output); **[Proposed]** problems have an empty cell for you: their solutions are discussed in the seminar.
- Models: A2 after A1; A4 after A3; A6 after A5; A8, A9 after A7; B4 after B2; B5 after B3; B6, B8 after B1; B7 after A5, A6; B9 after B1, A2.

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_4'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import io
import re
import json
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'
BNR_XML = 'https://curs.bnr.ro/files/xml/years/nbrfxrates{}.xml'

SECTORS = ['XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']        # din dec. 1998
SECTOR_NAMES = {'XLB': 'Materials', 'XLE': 'Energy', 'XLF': 'Financials', 'XLI': 'Industrials',
                'XLK': 'Technology', 'XLP': 'Cons. staples', 'XLU': 'Utilities', 'XLV': 'Health care',
                'XLY': 'Cons. discretionary'}
MULTI = ['SPY', 'EFA', 'EEM', 'TLT', 'IEF', 'LQD', 'HYG', 'GLD']                  # HYG din apr. 2007
MULTI_NAMES = {'SPY': 'US equities', 'EFA': 'Developed ex-US', 'EEM': 'Emerging markets',
               'TLT': 'US Treasuries 20y+', 'IEF': 'US Treasuries 7-10y', 'LQD': 'Investment-grade credit',
               'HYG': 'High-yield credit', 'GLD': 'Gold'}
BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'SNG', 'SNN', 'EL', 'TEL']                      # listate inainte de 2014-09
BVB_NAMES = {'TLV': 'Banca Transilvania', 'SNP': 'OMV Petrom', 'BRD': 'BRD-GSG', 'TGN': 'Transgaz',
             'SNG': 'Romgaz', 'SNN': 'Nuclearelectrica', 'EL': 'Electrica', 'TEL': 'Transelectrica'}

_CACHE = {}


def read_market(symbol):
    """Seria data/market/<SIMBOL>.csv."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def price(symbol):
    """Pretul folosit: adjusted_close pentru ETF/actiuni (.US, .RO), close pentru indici si cripto."""
    d = read_market(symbol)
    col = 'close' if symbol.endswith('.INDX') or symbol.endswith('.CC') or symbol == 'BET' else 'adjusted_close'
    s = d[col].astype(float)
    return s[s > 0].rename(symbol)


def prices(symbols, start=None, end=None):
    """Preturi aliniate: join pe zilele comune (analiza comuna -> intai join pe preturi)."""
    p = pd.concat([price(s) for s in symbols], axis=1).dropna()
    return p.loc[start:end]


def log_returns(p):
    """Randamente log pe un tabel deja aliniat (zile comune)."""
    return np.log(p).diff().dropna()


def french_rf():
    """Rata fara risc lunara (T-bill la o luna) din fisierul Fama-French cu 3 factori, in zecimal."""
    if 'rf' in _CACHE:
        return _CACHE['rf']
    url = FRENCH + 'F-F_Research_Data_Factors_CSV.zip'
    raw = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode('latin-1')
    rows = []
    for line in text.splitlines():
        parts = [x.strip() for x in line.split(',')]
        if len(parts) == 5 and len(parts[0]) == 6 and parts[0].isdigit():
            rows.append((parts[0], float(parts[4]) / 100.0))
        elif rows and not (len(parts[0]) == 6 and parts[0].isdigit()):
            break
    rf = pd.Series(dict(rows))
    rf.index = pd.to_datetime(rf.index, format='%Y%m') + pd.offsets.MonthEnd(0)
    _CACHE['rf'] = rf.rename('RF')
    return _CACHE['rf']


def monthly_returns(symbols, start=None):
    """Randamente lunare simple din preturile de sfarsit de luna (zile comune)."""
    p = prices(symbols).resample('ME').last()
    return p.pct_change().dropna().loc[start:]


def bnr_rate(currency='USD', start=2014, end=2026):
    """Cursul de referinta BNR: lei pentru o unitate de valuta (arhivele XML anuale)."""
    key = ('bnr', currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(start, end + 1):
        xml = urllib.request.urlopen(urllib.request.Request(BNR_XML.format(y), headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"( multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(2))))
    s = pd.Series(dict(rows)).sort_index()
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.rename(f'{currency}RON')
    return _CACHE[key]

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
PALETTE = [MainBlue, IDAred, Forest, Amber, Orange, Purple, Crimson, '#795548', '#17A2B8', '#6F42C1', '#20C997']
EWcol    = '#D63384'   # culoarea portofoliului 1/N (niciodata gri pentru serii)

SEED = 42
MAX_ABS_RET = 0.5        # prag pentru erori de date in seriile BVB (log-randament zilnic)


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


US_END = '2026-07-31'
BVB_END = '2026-08-31'
WINDOW = 60
STRATS = ['1/N', 'MV', 'MV-LO', 'GMV', 'GMV-LO', 'GMV-LW', 'ERC', 'HRP']
BVB_STRATS = ['1/N', 'MV-LO', 'GMV-LO', 'GMV-LW', 'ERC', 'HRP']
SCOL = {'1/N': EWcol, 'MV': IDAred, 'MV-LO': Orange, 'GMV': Amber, 'GMV-LO': Purple,
        'GMV-LW': MainBlue, 'ERC': Forest, 'HRP': '#17A2B8'}
COMBINED = SECTORS + [a for a in MULTI if a != 'SPY']
UNIVERSES = {'Sectors': SECTORS, 'Multi-asset': MULTI, 'Combined': COMBINED}
LW_BLOCKS = (1, 2, 4, 6, 8, 10)   # Ledoit-Wolf (2008), Section 3.2.2
LW_K = 5000   # pseudo sequences for the block calibration (Algorithm 3.1)
LW_M = 4999   # bootstrap draws for the p-value
LW_M_CAL = 499   # bootstrap draws inside the calibration
LW_SB_MEAN = 5   # mean block of the stationary bootstrap of the VAR(1) residuals


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def us_monthly(symbols, start=None, end=US_END):
    """Randamente lunare simple (totale) si in exces fata de RF, pe lunile comune."""
    r = monthly_returns([s + '.US' for s in symbols]).loc[start:end]
    r.columns = symbols
    rf = french_rf().reindex(r.index)
    ok = rf.notna()
    r, rf = r[ok], rf[ok]
    return r, r.sub(rf, axis=0), rf


def bvb_monthly(symbols=BVB, bench=('BETTR.INDX', 'BET'), end=BVB_END):
    """Randamente lunare BVB (RON) din randamente zilnice filtrate (|r| > 0.5 = eveniment neajustat)."""
    cols = [s + '.RO' for s in symbols] + list(bench)
    p = prices(cols)
    lr = log_returns(p)
    bad = lr.abs() > MAX_ABS_RET
    lr = lr.mask(bad, 0.0)
    m = np.exp(lr.resample('ME').sum()) - 1
    m = m.loc[:end].iloc[1:]                     # prima luna este incompleta
    m.columns = list(symbols) + ['BET-TR' if b.startswith('BETTR') else b for b in bench]
    removed = {c.replace('.RO', ''): [str(d.date()) for d in lr.index[bad[c]]] for c in cols if bad[c].any()}
    return m, removed


def lw_cc(X):
    """Ledoit-Wolf (2004): shrinkage spre tinta cu corelatie constanta. Intoarce (Sigma, delta)."""
    X = np.asarray(X, float)
    T, N = X.shape
    Y = X - X.mean(0)
    S = Y.T @ Y / T
    var = np.diag(S).reshape(-1, 1)
    sd = np.sqrt(var)
    rbar = (np.sum(S / (sd @ sd.T)) - N) / (N * (N - 1))
    F = rbar * (sd @ sd.T)
    np.fill_diagonal(F, np.diag(S))
    Y2 = Y ** 2
    pi_mat = Y2.T @ Y2 / T - S ** 2
    pi_hat = pi_mat.sum()
    gamma_hat = np.linalg.norm(S - F, 'fro') ** 2
    theta = (Y ** 3).T @ Y / T - np.tile(var, (1, N)) * S
    np.fill_diagonal(theta, 0.0)
    rho_hat = np.trace(pi_mat) + rbar * np.sum(((1 / sd) @ sd.T) * theta)
    kappa = (pi_hat - rho_hat) / gamma_hat
    delta = max(0.0, min(1.0, kappa / T))
    return delta * F + (1 - delta) * S, delta


def w_ew(n):
    """Portofoliul 1/N."""
    return np.ones(n) / n


def w_gmv(S):
    """Portofoliul de varianta minima globala: S^-1 1 / 1' S^-1 1."""
    x = np.linalg.solve(S, np.ones(len(S)))
    return x / x.sum()


def w_tan(mu, S):
    """Portofoliul tangent (Sharpe maxim) cu vanzari in lipsa: S^-1 mu / 1' S^-1 mu."""
    x = np.linalg.solve(S, mu)
    return x / x.sum()


def w_long_only(S, mu=None):
    """Varianta minima (mu=None) sau Sharpe maxim cu ponderi 0 <= w <= 1, suma 1 (SLSQP)."""
    n = len(S)
    if mu is None:
        f = lambda w: w @ S @ w
    else:
        f = lambda w: -(w @ mu) / np.sqrt(w @ S @ w)
    res = minimize(f, np.ones(n) / n, method='SLSQP', bounds=[(0, 1)] * n,
                   constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
                   options={'ftol': 1e-12, 'maxiter': 500})
    w = np.clip(res.x, 0, None)
    return w / w.sum()


def w_erc(S):
    """Contributii egale la risc (Maillard, Roncalli, Teiletche 2010): min 0.5 y'Sy - sum(log y)/n."""
    n = len(S)
    b = np.ones(n) / n
    f = lambda y: 0.5 * y @ S @ y - b @ np.log(y)
    grad = lambda y: S @ y - b / y
    y0 = 1 / np.sqrt(np.diag(S))
    res = minimize(f, y0 / y0.sum(), jac=grad, method='L-BFGS-B', bounds=[(1e-12, None)] * n,
                   options={'ftol': 1e-15, 'gtol': 1e-12, 'maxiter': 2000})
    return res.x / res.x.sum()


def hrp_tree(S):
    """Arborele HRP (Lopez de Prado 2016, Stage 1): d_ij = sqrt((1 - rho_ij)/2); apoi distanta euclidiana
    intre coloanele matricei D, d~_ij = sqrt(sum_n (d_ni - d_nj)^2); legatura simpla (single linkage) pe d~."""
    sd = np.sqrt(np.diag(S))
    C = S / np.outer(sd, sd)
    D = np.sqrt(np.clip((1 - C) / 2, 0, None))
    np.fill_diagonal(D, 0)
    return linkage(pdist(D, 'euclidean'), 'single')


def w_hrp(S):
    """Hierarchical Risk Parity (Lopez de Prado 2016): ordonare dupa arbore + bisectie recursiva."""
    S = np.asarray(S)
    order = list(leaves_list(hrp_tree(S)))
    w = np.ones(len(S))

    def cvar(idx):
        s = S[np.ix_(idx, idx)]
        iv = 1 / np.diag(s)
        iv /= iv.sum()
        return iv @ s @ iv

    clusters = [order]
    while clusters:
        nxt = []
        for c in clusters:
            if len(c) < 2:
                continue
            h = len(c) // 2
            a, b = c[:h], c[h:]
            va, vb = cvar(a), cvar(b)
            alpha = 1 - va / (va + vb)
            w[a] *= alpha
            w[b] *= 1 - alpha
            nxt += [a, b]
        clusters = nxt
    return w / w.sum()


def risk_contrib(w, S):
    """Contributiile procentuale la varianta: w_i (S w)_i / w'Sw."""
    w = np.asarray(w)
    return w * (S @ w) / (w @ S @ w)


def weights(name, Rw):
    """Ponderile strategiei `name` estimate pe fereastra Rw (randamente in exces, T x N)."""
    X = np.asarray(Rw, float)
    n = X.shape[1]
    mu = X.mean(0)
    S = np.cov(X.T)
    if name == '1/N':
        return w_ew(n)
    if name == 'MV':
        return w_tan(mu, S)
    if name == 'MV-LO':
        return w_long_only(S, mu)
    if name == 'GMV':
        return w_gmv(S)
    if name == 'GMV-LO':
        return w_long_only(S)
    if name == 'GMV-LW':
        return w_gmv(lw_cc(X)[0])
    if name == 'GMV-NL':
        return w_gmv(nl_shrink(X))
    if name == 'ERC':
        return w_erc(S)
    if name == 'HRP':
        return w_hrp(S)
    raise ValueError(name)


def backtest(R, Rex, window=WINDOW, strategies=STRATS):
    """Reechilibrare lunara: ponderi din ultimele `window` luni, aplicate in luna urmatoare.

    R: randamente totale, Rex: randamente in exces (acelasi index). Intoarce randamentele in exces
    ale portofoliilor, turnover-ul (suma |w_nou - w_derivat|) si istoricul ponderilor.
    """
    R, Rex = R.values, Rex
    idx = Rex.index[window:]
    X = Rex.values
    out = {s: [] for s in strategies}
    turn = {s: [] for s in strategies}
    W = {s: [] for s in strategies}
    prev = {s: None for s in strategies}
    for t in range(window, len(X)):
        est = X[t - window:t]
        for s in strategies:
            w = weights(s, est)
            if prev[s] is not None:
                turn[s].append(np.abs(w - prev[s]).sum())
            else:
                turn[s].append(np.nan)
            out[s].append(w @ X[t])
            gross = 1 + R[t]
            prev[s] = w * gross / (w @ gross)
            W[s].append(w)
    ret = pd.DataFrame(out, index=idx)
    ret.attrs['rf'] = pd.Series(R[window:, 0] - X[window:, 0], index=idx)   # rata fara risc (0 daca R = Rex)
    to = pd.DataFrame(turn, index=idx)
    Wd = {s: pd.DataFrame(np.array(W[s]), index=idx, columns=Rex.columns) for s in strategies}
    return ret, to, Wd


def sharpe(r, periods=12):
    r = np.asarray(r)
    return r.mean() / r.std(ddof=1) * np.sqrt(periods)


def summary(ret, to, costs=(0, 10, 50)):
    """Medie, volatilitate, Sharpe (anualizate, randamente in exces), turnover mediu lunar, scaderea maxima
    a averii din randamente totale (exces + rata fara risc), Sharpe net de costuri (bp)."""
    rows = {}
    rf = ret.attrs.get('rf', 0.0)
    for s in ret.columns:
        r = ret[s]
        d = dict(mean=r.mean() * 12, vol=r.std() * np.sqrt(12), sharpe=sharpe(r),
                 turnover=to[s].mean(), mdd=max_drawdown(r + rf))
        for c in costs[1:]:
            d[f'sharpe_{c}bp'] = sharpe(r - c / 1e4 * to[s].fillna(0))
        rows[s] = d
    return pd.DataFrame(rows).T


def max_drawdown(r):
    """Scaderea maxima a averii; o pierdere lunara >= 100% anuleaza averea (drawdown -100%)."""
    r = np.asarray(r, float)
    if (r <= -1).any():
        return -1.0
    w = np.r_[1.0, np.cumprod(1 + r)]                          # averea initiala 1 intra in maximul curent
    return float((w / np.maximum.accumulate(w) - 1).min())


def sr_diff_hac(r1, r2, lags=None):
    """Ledoit-Wolf (2008), Sectiunea 3.1: Delta = SR1 - SR2 (lunar), eroare standard prin metoda delta,
    covarianta HAC cu nucleul Bartlett (Newey-West) si lag floor(4 (T/100)^(2/9))."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    T = len(r1)
    m1, m2 = r1.mean(), r2.mean()
    g1, g2 = (r1 ** 2).mean(), (r2 ** 2).mean()
    d = m1 / np.sqrt(g1 - m1 ** 2) - m2 / np.sqrt(g2 - m2 ** 2)
    grad = np.array([g1 / (g1 - m1 ** 2) ** 1.5, -g2 / (g2 - m2 ** 2) ** 1.5,
                     -m1 / (2 * (g1 - m1 ** 2) ** 1.5), m2 / (2 * (g2 - m2 ** 2) ** 1.5)])
    Y = np.column_stack([r1 - m1, r2 - m2, r1 ** 2 - g1, r2 ** 2 - g2])
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    Psi = Y.T @ Y / T
    for l in range(1, lags + 1):
        G = Y[l:].T @ Y[:-l] / T
        Psi += (1 - l / (lags + 1)) * (G + G.T)
    se = np.sqrt(grad @ Psi @ grad / T)
    p = 2 * (1 - stats.norm.cdf(abs(d / se)))
    return d * np.sqrt(12), se * np.sqrt(12), p


def _lw_parts(r1, r2):
    """Delta = SR1 - SR2 (lunar), gradientul lui f si seriile y_t (LW 2008, ec. 1-5)."""
    m1, m2 = r1.mean(-1), r2.mean(-1)
    g1, g2 = (r1 ** 2).mean(-1), (r2 ** 2).mean(-1)
    v1, v2 = g1 - m1 ** 2, g2 - m2 ** 2
    d = m1 / np.sqrt(v1) - m2 / np.sqrt(v2)
    grad = np.stack([g1 / v1 ** 1.5, -g2 / v2 ** 1.5, -m1 / (2 * v1 ** 1.5), m2 / (2 * v2 ** 1.5)], -1)
    Y = np.stack([r1 - m1[..., None], r2 - m2[..., None], r1 ** 2 - g1[..., None], r2 ** 2 - g2[..., None]], -1)
    return d, grad, Y


def qs_kernel(x):
    """Nucleul Quadratic Spectral (Andrews 1991)."""
    x = np.asarray(x, float)
    out = np.ones_like(x)
    nz = x != 0
    z = 6 * np.pi * x[nz] / 5
    out[nz] = 25 / (12 * np.pi ** 2 * x[nz] ** 2) * (np.sin(z) / z - np.cos(z))
    return out


def hac_qs_pw(Y):
    """Covarianta HAC cu nucleul QS prealbit (Andrews & Monahan 1992), latime de banda automata
    (Andrews 1991, AR(1)), factorul T/(T-4) din LW (2008, ec. 5). Y: T x 4, centrat."""
    T, k = Y.shape
    X0, X1 = Y[:-1], Y[1:]
    A = np.linalg.lstsq(X0, X1, rcond=None)[0].T                # VAR(1) fara termen liber
    U, s, Vt = np.linalg.svd(A)                                  # Andrews-Monahan: valori singulare <= 0.97
    A = U @ np.diag(np.minimum(s, 0.97)) @ Vt
    E = X1 - X0 @ A.T
    n = len(E)
    rho = np.array([np.sum(E[1:, a] * E[:-1, a]) / np.sum(E[:-1, a] ** 2) for a in range(k)])
    sig2 = np.array([np.mean((E[1:, a] - rho[a] * E[:-1, a]) ** 2) for a in range(k)])
    num = np.sum(4 * rho ** 2 * sig2 ** 2 / (1 - rho) ** 8)
    den = np.sum(sig2 ** 2 / (1 - rho) ** 4)
    ST = 1.3221 * (num / den * n) ** 0.2
    Psi = E.T @ E / n
    for j in range(1, n):
        w = qs_kernel(j / ST)
        G = E[j:].T @ E[:-j] / n
        Psi += w * (G + G.T)
    B = np.linalg.inv(np.eye(k) - A)
    return T / (T - 4) * B @ Psi @ B.T


def lw_se(r1, r2):
    """Delta (lunar) si eroarea standard s(Delta) din datele originale (LW 2008, ec. 5)."""
    d, grad, Y = _lw_parts(np.asarray(r1, float), np.asarray(r2, float))
    Psi = hac_qs_pw(Y)
    return float(d), float(np.sqrt(grad @ Psi @ grad / len(r1)))


def cbb_stat(r1, r2, b, M, rng):
    """M replicari bootstrap circular pe blocuri de lungime b: Delta* si s(Delta*) 'natural'
    (Goetze & Kuensch 1996; LW 2008, Sectiunea 3.2.2)."""
    T = len(r1)
    nb = -(-T // b)
    st = rng.integers(0, T, (M, nb))
    idx = ((st[:, :, None] + np.arange(b)) % T).reshape(M, -1)[:, :T]
    x1, x2 = r1[idx], r2[idx]
    d, grad, Y = _lw_parts(x1, x2)
    l = T // b
    Z = Y[:, :l * b].reshape(M, l, b, 4).sum(2) / np.sqrt(b)
    Psi = np.einsum('mjk,mjl->mkl', Z, Z) / l
    se = np.sqrt(np.einsum('mk,mkl,ml->m', grad, Psi, grad) / T)
    return d, se


def _sb_indices(T, n, mean_block, rng):
    """Indicii unui bootstrap stationar (Politis & Romano 1994), bloc mediu `mean_block`."""
    idx = np.empty(n, int)
    idx[0] = rng.integers(T)
    new = rng.random(n) < 1 / mean_block
    starts = rng.integers(0, T, n)
    for t in range(1, n):
        idx[t] = starts[t] if new[t] else (idx[t - 1] + 1) % T
    return idx


def lw_block_calibration(r1, r2, blocks=LW_BLOCKS, K=LW_K, M=LW_M_CAL, alpha=0.05, seed=SEED):
    """Algoritmul 3.1 din LW (2008): VAR(1) + bootstrap stationar al reziduurilor, K secvente pseudo,
    acoperirea intervalului studentizat simetric pentru fiecare b; alege b cu acoperirea cea mai apropiata de 1-alpha."""
    rng = np.random.default_rng(seed)
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    X = np.column_stack([r1, r2])
    T = len(X)
    d0, _ = lw_se(r1, r2)
    Z = np.column_stack([np.ones(T - 1), X[:-1]])
    coef = np.linalg.lstsq(Z, X[1:], rcond=None)[0]
    U = X[1:] - Z @ coef
    U = U - U.mean(0)
    cover = np.zeros(len(blocks))
    for k in range(K):
        u = U[_sb_indices(len(U), T - 1, LW_SB_MEAN, rng)]
        Xs = np.empty_like(X)
        Xs[0] = X[0]
        for t in range(1, T):
            Xs[t] = coef[0] + Xs[t - 1] @ coef[1:] + u[t - 1]
        dk, sk = lw_se(Xs[:, 0], Xs[:, 1])
        for j, b in enumerate(blocks):
            ds, ss = cbb_stat(Xs[:, 0], Xs[:, 1], b, M, rng)
            z = np.quantile(np.abs(ds - dk) / ss, 1 - alpha)
            cover[j] += abs(dk - d0) <= z * sk
    g_hat = cover / K
    return blocks[int(np.argmin(np.abs(g_hat - (1 - alpha))))], dict(zip(map(str, blocks), g_hat.tolist()))


def sr_diff_boot(r1, r2, block=None, M=LW_M, K=LW_K, alpha=0.05, seed=SEED):
    """Testul Ledoit-Wolf (2008, Sectiunea 3.2.2 si Remarca 3.2) pentru H0: SR1 = SR2:
    bootstrap studentizat circular pe blocuri; blocul din Algoritmul 3.1 (daca block=None).
    Intoarce Delta (anualizat), intervalul 95% simetric (anualizat), valoarea p (ec. 9), statisticile
    studentizate bootstrap, blocul ales si functia de calibrare."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    cal = None
    if block is None:
        block, cal = lw_block_calibration(r1, r2, K=K, alpha=alpha, seed=seed)
    d, s = lw_se(r1, r2)
    ds, ss = cbb_stat(r1, r2, block, M, np.random.default_rng(seed + 1))
    tstar = (ds - d) / ss
    z = np.quantile(np.abs(tstar), 1 - alpha)
    p = (np.sum(np.abs(tstar) >= abs(d) / s) + 1) / (M + 1)
    a = np.sqrt(12)
    return dict(diff=d * a, se=s * a, ci=[(d - z * s) * a, (d + z * s) * a], p_boot=float(p),
                t_obs=d / s, z_star=float(z), block=int(block), calibration=cal), tstar


def _test_pair(args):
    """Un test de diferenta Sharpe: HAC (Sectiunea 3.1) si bootstrap studentizat (Sectiunea 3.2.2)."""
    r1, r2, block = args
    d, se, p = sr_diff_hac(r1, r2)
    res, tstar = sr_diff_boot(r1, r2, block=block)
    return dict(diff=d, se_hac=se, p_hac=p, ci_boot=res['ci'], p_boot=res['p_boot'], block=res['block'],
                calibration=res['calibration'], se_qs=res['se'], t_obs=res['t_obs'], z_star=res['z_star']), tstar


def run_tests(pairs, n_jobs=1, blocks=None):
    """pairs: dict {cheie: (r1, r2)}; blocks: dict {cheie: bloc} deja calibrat (altfel Algoritmul 3.1);
    n_jobs > 1 foloseste procese paralele (calibrarea e costisitoare)."""
    keys = list(pairs)
    blocks = blocks or {}
    args = [(np.asarray(pairs[k][0], float), np.asarray(pairs[k][1], float), blocks.get(k)) for k in keys]
    if n_jobs > 1:
        from concurrent.futures import ProcessPoolExecutor
        with ProcessPoolExecutor(n_jobs) as ex:
            res = list(ex.map(_test_pair, args))
    else:
        res = [_test_pair(a) for a in args]
    return dict(zip(keys, res))


def nl_shrink(X):
    """Shrinkage neliniar analitic (Ledoit & Wolf 2020, Annals of Statistics), cazul N <= T-1."""
    X = np.asarray(X, float)
    X = X - X.mean(0)
    n, p = X.shape
    n = n - 1                                               # corectie pentru centrare
    S = X.T @ X / n
    lam, U = np.linalg.eigh(S)
    lam = np.maximum(lam, 1e-18)
    L = np.tile(lam[:, None], (1, p))
    h = n ** (-1 / 3)
    H = h * L.T
    x = (L - L.T) / H
    ft = (3 / 4 / np.sqrt(5)) * np.mean(np.maximum(1 - x ** 2 / 5, 0) / H, axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        Hf = (-3 / 10 / np.pi) * x + (3 / 4 / np.sqrt(5) / np.pi) * (1 - x ** 2 / 5) * \
            np.log(np.abs((np.sqrt(5) - x) / (np.sqrt(5) + x)))
    edge = np.isclose(np.abs(x), np.sqrt(5))
    Hf[edge] = (-3 / 10 / np.pi) * x[edge]
    Hft = np.mean(Hf / H, axis=1)
    c = p / n
    d = lam / ((np.pi * c * lam * ft) ** 2 + (1 - c - np.pi * c * lam * Hft) ** 2)
    return U @ np.diag(d) @ U.T


def to_py(o):
    if isinstance(o, dict):
        return {str(k): to_py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_py(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return to_py(o.tolist())
    if isinstance(o, pd.DataFrame):
        return to_py(o.to_dict(orient='index'))
    return o


def save_fig(name):
    """Show the figure."""
    plt.show()
    plt.close()

In [ ]:
def two_asset_stats():
    R, Rex, rf = us_monthly(['SPY', 'TLT'], start='2002-08-01')
    mu = R.mean() * 12
    sd = R.std() * np.sqrt(12)
    rho = R.corr().iloc[0, 1]
    return R, mu, sd, rho


def gmv_two(s1, s2, rho):
    """Ponderea primului activ in GMV cu doua active."""
    return (s2 ** 2 - rho * s1 * s2) / (s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2)


def black_litterman(Sigma, w_b, P, q, delta=2.5, tau=0.05, conf=None):
    """Black-Litterman (1992), forma He-Litterman: Omega = diag(P tau Sigma P') daca nu e data."""
    pi = delta * Sigma @ w_b
    Om = np.diag(np.diag(P @ (tau * Sigma) @ P.T)) if conf is None else conf
    A = np.linalg.inv(tau * Sigma) + P.T @ np.linalg.inv(Om) @ P
    mu_bl = np.linalg.solve(A, np.linalg.inv(tau * Sigma) @ pi + P.T @ np.linalg.inv(Om) @ q)
    w_bl = np.linalg.solve(delta * Sigma, mu_bl)
    return pi, mu_bl, w_bl


def run_backtests():
    out = {}
    for name, syms in UNIVERSES.items():
        R, Rex, rf = us_monthly(syms)
        out[name] = backtest(R, Rex)
        print(f'   backtest {name}: {len(out[name][0])} months')
    return out


def sr_diff_iid(sr1, sr2, rho, T):
    """Diferenta de Sharpe sub randamente i.i.d. Normale (formula raportata de Ledoit-Wolf 2008).

    sr1, sr2: Sharpe lunar; rho: corelatia randamentelor; intoarce (z, p).
    """
    v = (2 - 2 * rho + 0.5 * (sr1 ** 2 + sr2 ** 2 - 2 * sr1 * sr2 * rho ** 2)) / T
    z = (sr1 - sr2) / np.sqrt(v)
    return z, 2 * (1 - stats.norm.cdf(abs(z))), np.sqrt(v)


def var_ratio_boot(r1, r2, block=6, B=5000, seed=SEED):
    """Raportul varianțelor out-of-sample var(r1)/var(r2), cu interval bootstrap circular pe blocuri."""
    rng = np.random.default_rng(seed)
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    T = len(r1)
    nb = int(np.ceil(T / block))
    e1, e2 = np.r_[r1, r1[:block]], np.r_[r2, r2[:block]]
    out = np.empty(B)
    for b in range(B):
        st = rng.integers(0, T, nb)
        idx = (st[:, None] + np.arange(block)).ravel()[:T]
        out[b] = e1[idx].var() / e2[idx].var()
    return r1.var() / r2.var(), np.percentile(out, [2.5, 97.5]), np.mean(out >= 1)


def britten_jones(Rex):
    """Britten-Jones (1999): regresia lui 1 pe randamentele in exces, fara termen liber.
    b ~ Sigma^-1 mu; testul t al lui b_i = 0 (pondere tangenta nula) si testul F al lui b proportional cu 1 (1/N)."""
    X = np.asarray(Rex, float)
    T, N = X.shape
    y = np.ones(T)
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    u = y - X @ b
    s2 = u @ u / (T - N)
    se = np.sqrt(np.diag(s2 * XtX_inv))
    t = b / se
    R = np.hstack([np.eye(N - 1), np.zeros((N - 1, 1))]) - np.hstack([np.zeros((N - 1, N - 1)), np.ones((N - 1, 1))])
    Rb = R @ b
    F = Rb @ np.linalg.solve(R @ XtX_inv @ R.T, Rb) / (N - 1) / s2
    pF = 1 - stats.f.cdf(F, N - 1, T - N)
    # varianta HAC (Bartlett, Newey-West), pentru abateri de la i.i.d. Normal
    L = int(np.floor(4 * (T / 100) ** (2 / 9)))
    Xu = X * u[:, None]
    Om = Xu.T @ Xu / T
    for l in range(1, L + 1):
        G = Xu[l:].T @ Xu[:-l] / T
        Om += (1 - l / (L + 1)) * (G + G.T)
    Vh = T * XtX_inv @ Om @ XtX_inv
    W = Rb @ np.linalg.solve(R @ Vh @ R.T, Rb)
    pW = 1 - stats.chi2.cdf(W, N - 1)
    w = b / b.sum()
    return dict(T=T, N=N, b=b.tolist(), w=w.tolist(), t=t.tolist(), F=float(F), pF=float(pF), df=(N - 1, T - N),
                wald_hac=float(W), p_wald_hac=float(pW), t_hac=(b / np.sqrt(np.diag(Vh))).tolist(),
                w_check=w_tan(X.mean(0), np.cov(X.T)).tolist())


def kz_bias(theta_ann, N, T):
    """Kan & Zhou (2007): E[theta_hat^2] = (T theta^2 + N)/(T - N - 2) pentru Sigma_hat de verosimilitate maxima,
    i.i.d. Normal; intoarce valorile anualizate."""
    th2 = (theta_ann / np.sqrt(12)) ** 2
    e2 = (T * th2 + N) / (T - N - 2)
    return dict(theta2_m=th2, E_theta2_m=e2, E_theta_ann_approx=float(np.sqrt(e2 * 12)),
                inv_bias=T / (T - N - 2))


def mp_bounds(c, s2=1.0):
    """Suportul legii Marchenko-Pastur pentru raportul c = N/T < 1."""
    return s2 * (1 - np.sqrt(c)) ** 2, s2 * (1 + np.sqrt(c)) ** 2


FOREIGN = ['SPY', 'EFA', 'IEF', 'GLD']
# backtests of the lecture, reused below
bts = run_backtests()

## Part A — pen-and-paper exercises (numerical check)

- The cells below check the numbers of A1–A9; the full statements (assumptions, sub-tasks, what to report) are in the Seminar 4 slides.

### A1. The GMV portfolio for N assets [Solved]

In [5]:
# A1 two-asset GMV: SPY and TLT
R, mu, sd, rho = two_asset_stats()
w = gmv_two(sd['SPY'], sd['TLT'], rho)
vol = np.sqrt(w**2 * sd['SPY']**2 + (1 - w)**2 * sd['TLT']**2 + 2 * w * (1 - w) * rho * sd['SPY'] * sd['TLT'])
print(f"sigma1 = {sd['SPY']:.4f}  sigma2 = {sd['TLT']:.4f}  rho = {rho:.4f}")
print(f"w* = {w:.3f}  vol = {vol:.4f}  mean = {w * mu['SPY'] + (1 - w) * mu['TLT']:.4f}")
Sig2 = np.array([[sd['SPY']**2, rho * sd['SPY'] * sd['TLT']], [rho * sd['SPY'] * sd['TLT'], sd['TLT']**2]])
A = np.ones(2) @ np.linalg.solve(Sig2, np.ones(2))
print(f'A = {A:.1f}  1/sqrt(A) = {1 / np.sqrt(A):.4f}  w_GMV = {(np.linalg.solve(Sig2, np.ones(2)) / A).round(3)}')

sigma1 = 0.1472  sigma2 = 0.1349  rho = -0.1005
w* = 0.461  vol = 0.0943  mean = 0.0773
A = 112.4  1/sqrt(A) = 0.0943  w_GMV = [0.461 0.539]


### A2. Tangency weights by regression [Proposed]

In [ ]:
# Your code here


### A3. ERC with two assets [Solved]

In [7]:
# A3 ERC with two assets = inverse volatility
r, rex, rf = us_monthly(['SPY', 'IEF'])
S2 = r.cov().values * 12
s = np.sqrt(np.diag(S2))
w_iv = (1 / s) / (1 / s).sum()
print('inverse volatility:', w_iv.round(3), ' numerical ERC:', w_erc(S2).round(3))
print('risk shares:', risk_contrib(w_iv, S2).round(3))

inverse volatility: [0.311 0.689]  numerical ERC: [0.311 0.689]
risk shares: [0.5 0.5]


### A4. Who carries the risk in 60/40? [Proposed]

In [ ]:
# Your code here


### A5. Black–Litterman as GLS [Solved]

In [9]:
# A5 precision-weighted average
pi, q, tau, sig, om = 0.05, 0.08, 0.05, 0.16, 0.02**2
pp, pv = 1 / (tau * sig**2), 1 / om
print(f'precisions {pp:.0f} {pv:.0f};  mu_BL = {(pp * pi + pv * q) / (pp + pv):.4f};  '
      f'view weight = {pv / (pp + pv):.3f};  sd = {np.sqrt(1 / (pp + pv)):.4f}')

precisions 781 2500;  mu_BL = 0.0729;  view weight = 0.762;  sd = 0.0175


### A6. The sector view of the lecture [Proposed]

In [ ]:
# Your code here


### A7. The variance of a Sharpe difference [Solved]

In [11]:
# A7 i.i.d. test of a Sharpe ratio difference
ret = bts['Sectors'][0]
sr1, sr2 = ret['GMV'].mean() / ret['GMV'].std(), ret['1/N'].mean() / ret['1/N'].std()
rho = ret[['GMV', '1/N']].corr().iloc[0, 1]
z, p, se = sr_diff_iid(sr1, sr2, rho, len(ret))
print(f'T = {len(ret)}  SR = {sr1:.4f}, {sr2:.4f}  rho = {rho:.4f}  se = {se:.4f}  z = {z:.2f}  p = {p:.3f}')
print('HAC p =', round(sr_diff_hac(ret['GMV'], ret['1/N'])[2], 3))

T = 271  SR = 0.2051, 0.1908  rho = 0.6939  se = 0.0483  z = 0.30  p = 0.766
HAC p = 0.754


### A8. ERC versus 1/N [Proposed]

In [ ]:
# Your code here


### A9. The in-sample Sharpe bias [Proposed]

In [ ]:
# Your code here


## Part B — backtests and inference

### B1. 1/N versus Markowitz on sectors [Solved]

- Interpretation: MV-LO beats 1/N by 0.06 with $p = 0.60$; how long a sample would make this difference significant?

In [14]:
SEM_BLOCKS = {'MV': 6, 'MV-LO': 1, 'GMV': 1, 'HRP-GMV': 1}   # calibrated block lengths (Algorithm 3.1)
# Solution
ret, to, W = bts['Sectors']
for s in ['MV', 'MV-LO', 'GMV']:
    d, se, p = sr_diff_hac(ret[s], ret['1/N'])
    rb, _ = sr_diff_boot(ret[s], ret['1/N'], block=SEM_BLOCKS[s])
    print(f"{s:6s} Delta = {d:+.3f}  se = {se:.3f}  p HAC = {p:.3f}  bootstrap interval = {np.round(rb['ci'], 3)}  p boot = {rb['p_boot']:.3f}  block = {rb['block']}")
summary(ret, to).loc[['1/N', 'MV', 'MV-LO', 'GMV']].round(3)

MV     Delta = -0.404  se = 0.331  p HAC = 0.223  bootstrap interval = [-1.175  0.367]  p boot = 0.296  block = 6
MV-LO  Delta = +0.063  se = 0.120  p HAC = 0.597  bootstrap interval = [-0.187  0.314]  p boot = 0.627  block = 1
GMV    Delta = +0.050  se = 0.159  p HAC = 0.754  bootstrap interval = [-0.269  0.369]  p boot = 0.762  block = 1


,mean,vol,sharpe,turnover,mdd,sharpe_10bp,sharpe_50bp
1/N,0.094,0.143,0.661,0.024,-0.491,0.659,0.651
MV,0.178,0.690,0.258,3.855,-1.000,0.194,-0.080
MV-LO,0.107,0.147,0.724,0.235,-0.423,0.705,0.629
GMV,0.085,0.119,0.711,0.234,-0.326,0.687,0.594


### B2. Does shrinkage lower GMV risk? [Solved]

- Interpretation: why is the gain small with 9 assets and 60 months? When would it be larger?

In [15]:
# Solution
vr, ci, share = var_ratio_boot(ret['GMV-LW'], ret['GMV'])
print(f"vol GMV = {ret['GMV'].std() * np.sqrt(12):.4f}  vol GMV-LW = {ret['GMV-LW'].std() * np.sqrt(12):.4f}")
print(f'variance ratio = {vr:.3f}  95% = {ci.round(3)}')
print(f"turnover: GMV {to['GMV'].mean():.3f}  GMV-LW {to['GMV-LW'].mean():.3f}")
# (d) Marchenko-Pastur
ev = np.sort(np.linalg.eigvalsh(np.corrcoef(us_monthly(SECTORS)[1].values[-WINDOW:].T)))[::-1]
lo, hi = mp_bounds(9 / WINDOW)
print('eigenvalues', ev.round(3), ' band', round(lo, 3), round(hi, 3), ' above', (ev > hi).sum(), ' below', (ev < lo).sum())

vol GMV = 0.1190  vol GMV-LW = 0.1180
variance ratio = 0.984  95% = [0.918 1.062]
turnover: GMV 0.234  GMV-LW 0.137


eigenvalues [5.246 1.349 0.835 0.499 0.32  0.29  0.212 0.143 0.107]  band 0.375 1.925  above 1  below 5


### B3. ERC and HRP on sectors [Solved]

- Interpretation: ERC and 1/N have correlation 0.993; is ERC a different strategy or a small tilt?

In [16]:
# Solution
X = us_monthly(SECTORS)[1].values[-WINDOW:]
S9 = np.cov(X.T)
print(pd.DataFrame({'ERC': w_erc(S9), 'HRP': w_hrp(S9), '1/N risk share': risk_contrib(w_ew(9), S9)},
                   index=SECTORS).round(3))
for s in ['ERC', 'HRP']:
    d, se, p = sr_diff_hac(ret[s], ret['1/N'])
    print(f"{s}: SR = {sharpe(ret[s]):.3f}  Delta = {d:+.3f}  p = {p:.3f}  corr(1/N) = {ret[[s, '1/N']].corr().iloc[0, 1]:.4f}  turnover = {to[s].mean():.3f}")

       ERC    HRP  1/N risk share
XLB  0.086  0.045           0.138
XLE  0.109  0.093           0.108
XLF  0.106  0.100           0.114
XLI  0.090  0.050           0.134
XLK  0.102  0.111           0.121
XLP  0.148  0.184           0.078
XLU  0.125  0.171           0.093
XLV  0.139  0.152           0.083
XLY  0.095  0.094           0.131
ERC: SR = 0.699  Delta = +0.038  p = 0.091  corr(1/N) = 0.9927  turnover = 0.028
HRP: SR = 0.700  Delta = +0.039  p = 0.325  corr(1/N) = 0.9789  turnover = 0.082


### B4. Shrinkage on 16 ETFs [Proposed]

- Interpretation: is constant correlation a sensible target for stocks and Treasuries together?

In [ ]:
# Your code here


### B5. Risk parity with Bitcoin [Proposed]

- Interpretation: would you expect the past Bitcoin premium to persist? What about survivorship?

In [ ]:
# Your code here


### B6. HRP versus GMV on 16 ETFs [Proposed]

- Interpretation: GMV wins on volatility, HRP on Sharpe; which criterion fits which investor?

In [ ]:
# Your code here


### B7. Black–Litterman on a multi-asset set [Proposed]

- Interpretation: the weights move far although the view is modest; why?

In [ ]:
# Your code here


### B8. The break-even cost [Proposed]

- Interpretation: retail costs on BVB or US ETFs; is MV-LO worth implementing?

In [ ]:
# Your code here


### B9. Is the tangency portfolio 1/N? [Proposed]

- Interpretation: when the two tests disagree, which one do you trust with T = 166 and 8 restrictions?

In [ ]:
# Your code here


## Part C — an optimal portfolio of BVB blue chips in RON: does diversification beyond BET pay? [Proposed]

- Eight blue chips with and without SPY, EFA, IEF, GLD converted to RON at the BNR USD/RON reference rate; 36-month window; benchmark BET-TR.
- The reference results use R_f = 0: the Sharpe ratios are mean over volatility of RON returns; (d) asks for the effect of a RON risk-free rate.
- Discuss the RON risk-free rate, currency risk, costs and taxes.

In [ ]:
# Your code here
